In [ ]:
import seaborn as sns

BASE_FONTSIZE = 22
sns.set_theme(
    style="ticks",
    rc={
        "font.size": BASE_FONTSIZE,
        "axes.titlesize": BASE_FONTSIZE,
        "axes.labelsize": BASE_FONTSIZE,
        "xtick.labelsize": BASE_FONTSIZE * 0.85,
        "ytick.labelsize": BASE_FONTSIZE * 0.85,
        "legend.title_fontsize": BASE_FONTSIZE * 0.9,
        "legend.fontsize": BASE_FONTSIZE * 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "hatch.linewidth": 1.5,
        "hatch.color": "#333333",
    },
)


def apply_hatches(g, hatch_map):
    """Apply hatches by matching legend colors to bars."""
    # Build color -> hatch from legend
    color_to_hatch = {}
    for text, handle in zip(g._legend.get_texts(), g._legend.legend_handles):
        hatch = hatch_map.get(text.get_text(), None)
        if hatch:
            fc = handle.get_facecolor()
            color_to_hatch[tuple(fc)] = hatch

    # Apply to all bars
    for ax in g.axes.flat:
        for bar in ax.patches:
            hatch = color_to_hatch.get(tuple(bar.get_facecolor()))
            if hatch:
                bar.set_hatch(hatch)
                bar.set_edgecolor("white")

## Figures 1-3. Error terms

Complete the following two steps, then run the cells below to plot the results

1. Download or create necessary checkpoints
    ```sh

    # Cross terms and Correlation terms:
    uvx gdown 1JIeMHqJuA8fcV1WZOq67WVlcausxPoS5 -O downloads/checkpoints-analysis.tar.gz
    tar -xzvf downloads/checkpoints-analysis.tar.gz -C artifacts
    # or 
    # bash scripts/vision/finetune_analysis.sh

    # Drift terms:
    uvx gdown <URL> -O downloads/checkpoints-analysis-drift.tar.gz
    tar -xzvf downloads/checkpoints-analysis-drift.tar.gz artifacts/
    # or 
    # bash scripts/vision/finetune_analysis_drift.sh
    # bash scripts/vision/covariance-drifts.sh
    ```

2. Compute error terms
    ```sh
    python scripts/vision/generate_error_terms.py
    python scripts/vision/generate_error_terms_drift.py
    ```

In [ ]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

model = "ViT-B-16"
results_dir = f"../artifacts/results-analysis/{model}"

for type in ["cross", "corr"]:

    # Compute angular distance: https://en.wikipedia.org/wiki/Cosine_similarity
    df = pd.read_csv(os.path.join(results_dir, "error_terms.csv"))
    df["angular_distance"] = np.arccos(df["cosine_similarity"].clip(-1.0, 1.0)) / np.pi
    df = df[df["type"] == type]

    # Make violin plot
    plt.figure()
    sns.catplot(
        data=df, x="dataset", y="angular_distance", kind="violin", height=4, aspect=1.5
    )
    plt.ylim(0, 1)  # Set y-axis range from 0 to 1
    plt.ylabel(r"$\measuredangle$")
    plt.xlabel("Dataset")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(
        os.path.join(results_dir, f"error_{type}_term.pdf"), bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

model = "ViT-B-16"
results_dir = f"../artifacts/results-analysis-drift/{model}"

# Compute angular distance: https://en.wikipedia.org/wiki/Cosine_similarity
df = pd.read_csv(os.path.join(results_dir, "error_terms.csv"))
df["angular_distance"] = np.arccos(df["cosine_similarity"].clip(-1.0, 1.0)) / np.pi

# Make line plot
# plt.figure()
g = sns.relplot(
    data=df,
    x="step",
    y="angular_distance",
    hue="dataset",
    kind="line",
    height=4,
    aspect=1.5,
)
sns.move_legend(g, "upper center", ncol=3, bbox_to_anchor=(0.5, 1.05), title="Dataset")
plt.ylim(0, 1)  # Set y-axis range from 0 to 1
plt.ylabel(r"$\measuredangle$")
plt.xlabel("Iteration")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(results_dir, f"error_drift_term.pdf"), bbox_inches="tight")
plt.show()
plt.close()

## Figure 4. Performance on ViT/T5

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import seaborn as sns

methods = [
    # Baselines
    "zeroshot",
    "experts",
    "fisher",
    "ties",
    "sum04",
    "regmean",
    # "regmean-w",
    "mean",
    "isoc",
    # "isoc2",
    # "isoc3",
    "tsv",
    # "ace",
    "actmat",
    # "tact",
]

# Model identifiers as they appear on disk (artifacts/results/{model}-{method}/...).
# T5 dirs are lowercase to match the HF model id used by the language pipeline;
# ViT dirs preserve the OpenCLIP casing.
models = ["ViT-B-16", "ViT-B-32", "ViT-L-14", "t5-base", "t5-large", "Olmo-3-7b"]

model_to_model = {
    "ViT-B-16": "ViT-B-16",
    "ViT-B-32": "ViT-B-32",
    "ViT-L-14": "ViT-L-14",
    "t5-base": "T5-Base",
    "t5-large": "T5-Large",
    "Olmo-3-7b": "Olmo-3-7b",
}

method_to_method = {
    "zeroshot": "Zero-shot",
    "experts": "Experts",
    "regmean": "RegMean",
    "regmean-w": "RegMean-W",
    "fisher": "Fisher",
    "sum04": "TA ($\\alpha=0.4$)",
    "mean": "Average",
    "isoc": "Iso-C ($\\alpha=1.0$)",
    "isoc2": "Iso-C ($\\alpha=2.0$)",
    "isoc3": "Iso-C ($\\alpha=3.0$)",
    "tsv": "TSV",
    "ties": "TIES",
    "actmat": "ACTMat",
    "tact": "TACT",
    "ace": "ACE",
}

method_to_color = {
    # Simple baselines
    "TA": "#E2E8F0",
    "Average": "#A0AEC0",
    "Fisher": "#D8DEE6",
    "RegMean": "#CBD5E0",
    "TIES": "#B5C3D2",
    # SVD baselines
    "TSV": "#4A5568",
    "Iso-C ($\\alpha=1.0$)": "#8896A8",
    "Iso-C ($\\alpha=2.0$)": "#AAB4C0",
    "Iso-C ($\\alpha=3.0$)": "#C8CFD8",
    "KNOTS-TSV": "#4A5568",
    "KNOTS-Iso-C": "#718096",
    # ACTMat
    "ACTMat": "#00A658",
    "TACT": "#007A3D",
}

method_to_hatch = {
    "TA": "//",
    "RegMean": "//",
    "RegMean-W": "//",
    "Fisher": "//",
    "TIES": "//",
}

method_to_linestyle = {
    "Experts": "-",
    "Zero-shot": ".",
}

# Keys use the on-disk model id (so they match df["model"] before we remap to
# display names below).
model_to_group = {
    "t5-base::fft": "Language (FFT)",
    "t5-large::fft": "Language (FFT)",
    "ViT-B-16::fft": "Vision (FFT)",
    "ViT-B-32::fft": "Vision (FFT)",
    "ViT-L-14::fft": "Vision (FFT)",
    "Olmo-3-7b::fft": "RL (FFT)",
    # LoRA
    "t5-base::lora": "Language (LoRA)",
    "t5-large::lora": "Language (LoRA)",
    "ViT-B-16::lora": "Vision (LoRA)",
    "ViT-B-32::lora": "Vision (LoRA)",
    "ViT-L-14::lora": "Vision (LoRA)",
    "Olmo-3-7b::lora": "RL (LoRA)",
}

rows = []
for model in models:
    for method in methods:

        fft_filename = f"../artifacts/results/{model}-{method}/metrics.json"
        lora_filename = f"../artifacts/results/{model}-{method}/lora_metrics.json"

        for finetuning_mode, ft_filename in [
            ("fft", fft_filename),
            ("lora", lora_filename),
        ]:
            if not os.path.exists(ft_filename):
                continue
            with open(ft_filename, "r") as f:
                metrics = json.load(f)
                scores = [t["metrics"]["primary_score"] for t in metrics["tasks"]]
                avg_score = np.mean(scores)
                rows.append(
                    {
                        "model": model,
                        "finetuning_mode": finetuning_mode,
                        "method": method,
                        "acc": avg_score,
                    }
                )


df = pd.DataFrame(rows)
df["group"] = (df["model"] + "::" + df["finetuning_mode"]).map(
    lambda x: model_to_group[x]
)
df["model"] = df["model"].map(lambda x: model_to_model[x])
df["method"] = df["method"].map(lambda x: method_to_method[x])
# display(df)
g = sns.catplot(
    data=df,
    x="model",
    hue="method",
    y="acc",
    col="group",
    col_order=["Language (FFT)", "Language (LoRA)", "Vision (FFT)", "Vision (LoRA)"],
    kind="bar",
    palette=method_to_color,
    sharex=False,
)
# Formatting.
g.set_axis_labels("", "Accuracy (%)")
g.set_xticklabels(rotation=45)
g.set_titles("{col_name}", fontsize=BASE_FONTSIZE)
apply_hatches(g, method_to_hatch)

In [ ]:
# Display all methods
# filter (fft only)
dfp = df[df["model"] != "Olmo-3-7b"]
dfp = df[df["finetuning_mode"] == "fft"]
dfp = dfp.pivot(index="method", columns="model", values="acc")
dfp = (dfp * 100).round(1)
dfp.sort_values(by="ViT-B-16")

In [ ]:
# Display all methods
# filter (lora only)
dfp = df[df["model"] != "Olmo-3-7b"]
dfp = df[df["finetuning_mode"] == "lora"]
dfp = dfp.pivot(index="method", columns="model", values="acc")
dfp = (dfp * 100).round(1)
dfp.sort_values(by="ViT-B-16")

In [ ]:
# Display only RegMean and RegMean-W
dfp = df.pivot(index="method", columns="model", values="acc")
dfp = (dfp * 100).round(1)
dfp = dfp[dfp.index.isin(["RegMean", "RegMean-W"])]
dfp.sort_values(by="ViT-B-16")

In [ ]:
dfp = df[df["model"] == "Olmo-3-7b"]
dfp

## Table. Performance on OLMo

In [ ]:
import os.path as osp
import pandas as pd

task_to_metrics = {
    "codex_humaneval::tulu": ["pass_at_1", "pass_at_10"],
    "codex_humanevalplus::tulu": ["pass_at_1", "pass_at_10"],
    "ifeval::tulu": ["prompt_level_loose_acc"],
    "aime:zs_cot_r1::pass_at_32_2024_deepseek": ["pass_at_1", "pass_at_32"],
    "aime:zs_cot_r1::pass_at_32_2025_deepseek": ["pass_at_1", "pass_at_32"],
}

task_to_task = {
    "codex_humaneval::tulu": "HE",
    "codex_humanevalplus::tulu": "HE+",
    "ifeval::tulu": "IF",
    "aime:zs_cot_r1::pass_at_32_2024_deepseek": "A24",
    "aime:zs_cot_r1::pass_at_32_2025_deepseek": "A25",
}

metric_to_metric = {
    "pass_at_1": "@1",
    "pass_at_10": "@10",
    "pass_at_32": "@32",
    "prompt_level_loose_acc": "@1",
}

rows = []
methods = ["actmat", "tact"]
for method in methods:

    # Load metrics file
    path = f"../artifacts/results/Olmo-3-7b-{method}/metrics.json"
    if not osp.exists(path):
        continue
    with open(path) as f:
        data = json.load(f)

    for task in data["tasks"]:
        for metric in task_to_metrics[task["alias"]]:
            rows.append(
                {
                    "method": method,
                    "task": task_to_task[task["alias"]],
                    "metric": metric_to_metric[metric],
                    "score": task["metrics"][metric],
                }
            )
df = pd.DataFrame(rows)
df["task"] = df["task"] + df["metric"]

In [ ]:
dfp = df.pivot(index="method", columns="task", values="score")
column_order = [
    "HE@1",
    "HE@10",
    "HE+@1",
    "HE+@10",
    "A24@1",
    "A24@32",
    "A25@1",
    "A25@32",
    "IF@1",
]
dfp = (dfp * 100)[column_order].round(1)
dfp
# ACTMAT 58.2 83.7 54.5 79.8 39.9 80.0 29.8 66.7 47.0

## Figure 5. Correlation between activations and gradients

First run this script to generate the correlations: [scripts/vision/correlation.sh](../scripts/vision/correlation.sh)

In [ ]:
import sys
import os
import os.path as osp
from tqdm import tqdm
import torch
import pandas as pd
import torch.nn.functional as F

rootdir = "../artifacts/checkpoints/ViT-B-16"
datasets = [
    "Cars",
    "DTD",
    "EuroSAT",
    "GTSRB",
    "MNIST",
    "RESISC45",
    "SUN397",
    "SVHN",
]
ignore_keys = ["ln_", "ls_", "conv", "patch_dropout"]


def corr(x, y):
    if x.std() > 0 and y.std() > 0:
        return torch.corrcoef(torch.stack([x, y]))[0, 1]
    return torch.tensor(0.0)


rows = []
for dataset in tqdm(datasets, desc="Computing correlation coefficients"):
    corr_filepath = osp.join(rootdir, dataset + "Val", "correlation.pt")
    if not osp.exists(corr_filepath):
        continue
    cdict = torch.load(corr_filepath)
    for layer_name, layer_stats_dict in cdict.items():
        g, aat = layer_stats_dict["g_sq"], layer_stats_dict["aat_samples"]
        N_samples, N_indices = aat.shape
        for entry_idx in range(N_indices):
            rho = torch.abs(corr(g, aat[:, entry_idx])).item()
            rows.append(
                {
                    "dataset": dataset,
                    "layer": layer_name,
                    "entry_idx": entry_idx,
                    "rho": rho,
                }
            )
df = pd.DataFrame(rows)

In [ ]:
results_dir = f"../artifacts/results-analysis/ViT-B-16"
os.makedirs(results_dir, exist_ok=True)

df = df.rename(columns={"dataset": "Dataset", "rho": r"$\rho$"})
g = sns.catplot(data=df, x="Dataset", y=r"$\rho$", kind="violin", height=4, aspect=1.5)
g.set_xticklabels(rotation=45, ha="right")
g.savefig(
    os.path.join(results_dir, "activation_gradient_correlation.pdf"),
    bbox_inches="tight",
)

## Figure 6. Layer-wise cosine similarity

Download or create necessary checkpoints (same as figures 1-3)
```sh
# Download ckpts:
uvx gdown <URL> -O downloads/checkpoints.tar.gz
tar -xzvf downloads/checkpoints.tar.gz artifacts/
# or finetune and run eval to generate the covariance.pt files for each task
# bash scripts/vision/finetune.sh
# bash scripts/vision/eval_task_addition.sh # this creates the covariances on the fly for regmean
```

In [ ]:
import sys
import os.path as osp
from tqdm import tqdm
import torch
import pandas as pd
import torch.nn.functional as F

sys.path.append("..")
from src.vision.task_vectors import NonLinearTaskVector

rootdir = "../artifacts/checkpoints/ViT-B-16"
datasets = [
    "Cars",
    "DTD",
    "EuroSAT",
    "GTSRB",
    "MNIST",
    "RESISC45",
    "SUN397",
    "SVHN",
]

methods = {
    "ACTMat": lambda _, d: d.T @ d,
    "Identity": lambda c, _: torch.eye(
        c.shape[0], c.shape[1], dtype=c.dtype, device=c.device
    ),
}

rows = []
for dataset in tqdm(datasets, desc="Computing cosine similarities"):
    checkpoint_dir = osp.join(rootdir, dataset + "Val")
    tv = NonLinearTaskVector(checkpoint_dir=checkpoint_dir)
    cdict = torch.load(tv.covariance_path, map_location="cpu", weights_only=False)

    for k, d in tv.vector.items():
        kc = tv.param_key_to_cov_key(k)
        if kc not in cdict:
            continue

        for method_name, method_func in methods.items():
            c = cdict[kc]
            chat = method_func(c, d.to(c.device))
            rows.append(
                {
                    "dataset": dataset,
                    "layer": k,
                    "method": method_name,
                    "cosine_similarity": F.cosine_similarity(
                        c.reshape(-1), chat.reshape(-1), dim=0
                    ).item(),
                }
            )

# Create df
df = pd.DataFrame(rows)

In [ ]:
results_dir = f"../artifacts/results-analysis/ViT-B-16"
os.makedirs(results_dir, exist_ok=True)

df = df.rename(
    columns={
        "cosine_similarity": "Cos. Similarity",
        "dataset": "Dataset",
        "method": "Method",
    }
)

g = sns.catplot(
    data=df,
    x="Dataset",
    y="Cos. Similarity",
    hue="Method",
    kind="bar",
    height=4,
    aspect=1.5,
    hue_order=["Identity", "ACTMat"],
    palette={"Identity": "#E2E8F0", "ACTMat": "#00A658"},
)
g.set_xticklabels(rotation=45, ha="right")
g.savefig(
    os.path.join(results_dir, "covariance_estimator_cosine_similarity.pdf"),
    bbox_inches="tight",
)